# Statistical Correlation Analysis: Weather Sensitivity for Mobile Car Detailing

This notebook creates a synthetic 12-month operations dataset for Gloss & Grit Specialty Detailing. The dataset is designed to analyze how rainfall and extreme heat affect cancellation behavior for Ceramic Coating and Interior Restoration services.

In [1]:
import pandas as pd
import numpy as np

## Set up the date range

We create one full year of daily operations data.

In [2]:
np.random.seed(42)

dates = pd.date_range(start="2024-01-01", end="2024-12-31", freq="D")
len(dates)

366

## Create daily weather patterns

Rainfall and temperature are simulated with seasonal logic. Summer months have higher temperatures, while rainfall varies across the year.

In [3]:
weather_rows = []

for date in dates:
    month = date.month

    # Seasonal temperature baseline
    if month in [12, 1, 2]:
        base_temp = np.random.normal(52, 8)
    elif month in [3, 4, 5]:
        base_temp = np.random.normal(68, 9)
    elif month in [6, 7, 8]:
        base_temp = np.random.normal(90, 8)
    else:
        base_temp = np.random.normal(72, 9)

    max_temp = round(np.clip(base_temp, 35, 108), 1)

    # Rainfall probability by season
    if month in [3, 4, 5]:
        rain_chance = 0.38
    elif month in [6, 7, 8]:
        rain_chance = 0.26
    elif month in [9, 10, 11]:
        rain_chance = 0.30
    else:
        rain_chance = 0.32

    if np.random.random() < rain_chance:
        rainfall = round(np.random.gamma(shape=2.1, scale=4.2), 1)
    else:
        rainfall = 0.0

    weather_rows.append({
        "Date": date,
        "Daily_Rainfall_mm": rainfall,
        "Max_Temperature_F": max_temp
    })

weather_df = pd.DataFrame(weather_rows)
weather_df.head()

,Date,Daily_Rainfall_mm,Max_Temperature_F
0,2024-01-01,0.0,56.0
1,2024-01-02,0.0,50.9
2,2024-01-03,6.2,50.1
3,2024-01-04,12.6,64.6
4,2024-01-05,10.9,48.2


## Create multiple daily bookings

Each day can have several bookings for both service types. Ceramic Coating is more sensitive to rainfall, while Interior Restoration is more sensitive to extreme heat.

In [4]:
service_types = ["Ceramic Coating", "Interior Restoration"]

rows = []
booking_id = 1

for _, weather in weather_df.iterrows():
    date = weather["Date"]
    rain = weather["Daily_Rainfall_mm"]
    temp = weather["Max_Temperature_F"]
    weekday = date.day_name()
    month = date.month

    for service in service_types:

        # Simulate realistic daily booking volume
        if service == "Ceramic Coating":
            base_bookings = np.random.poisson(4)
        else:
            base_bookings = np.random.poisson(5)

        # Slightly higher demand on weekends
        if weekday in ["Saturday", "Sunday"]:
            base_bookings += np.random.randint(1, 4)

        # Slightly lower demand during heavy rain
        if rain >= 15 and service == "Ceramic Coating":
            base_bookings = max(1, base_bookings - np.random.randint(1, 3))

        bookings_today = max(1, base_bookings)

        for _ in range(bookings_today):

            # Base cancellation probability
            cancel_prob = 0.04

            if service == "Ceramic Coating":
                # Exterior service: strongly affected by rain
                if rain > 0:
                    cancel_prob += 0.07
                if rain >= 5:
                    cancel_prob += 0.10
                if rain >= 15:
                    cancel_prob += 0.18
                if rain >= 25:
                    cancel_prob += 0.12

                # Extreme heat has smaller but real effect
                if temp > 90:
                    cancel_prob += 0.04
                if temp > 100:
                    cancel_prob += 0.05

            else:
                # Interior service: less affected by rain
                if rain >= 10:
                    cancel_prob += 0.04
                if rain >= 25:
                    cancel_prob += 0.05

                # Interior service: more affected by heat
                if temp > 90:
                    cancel_prob += 0.10
                if temp > 98:
                    cancel_prob += 0.12
                if temp > 103:
                    cancel_prob += 0.08

            # Small random operational variation
            cancel_prob += np.random.normal(0, 0.015)
            cancel_prob = float(np.clip(cancel_prob, 0.02, 0.85))

            status = "Canceled" if np.random.random() < cancel_prob else "Completed"

            rows.append({
                "Booking_ID": f"B{booking_id:05d}",
                "Date": date,
                "Service_Type": service,
                "Daily_Rainfall_mm": rain,
                "Max_Temperature_F": temp,
                "Booking_Status": status,
                "Cancellation_Probability": round(cancel_prob, 3)
            })

            booking_id += 1

df = pd.DataFrame(rows)
df.head(10)

,Booking_ID,Date,Service_Type,Daily_Rainfall_mm,Max_Temperature_F,Booking_Status,Cancellation_Probability
0,B00001,2024-01-01,Ceramic Coating,0.0,56.0,Completed,0.047
1,B00002,2024-01-01,Ceramic Coating,0.0,56.0,Completed,0.030
2,B00003,2024-01-01,Ceramic Coating,0.0,56.0,Completed,0.058
3,B00004,2024-01-01,Interior Restoration,0.0,56.0,Completed,0.032
4,B00005,2024-01-01,Interior Restoration,0.0,56.0,Completed,0.049
5,B00006,2024-01-01,Interior Restoration,0.0,56.0,Completed,0.063
6,B00007,2024-01-01,Interior Restoration,0.0,56.0,Completed,0.067
7,B00008,2024-01-01,Interior Restoration,0.0,56.0,Completed,0.031
8,B00009,2024-01-01,Interior Restoration,0.0,56.0,Completed,0.034
9,B00010,2024-01-01,Interior Restoration,0.0,56.0,Completed,0.044


## Create Tableau helper fields

These fields make the Tableau analysis easier and help satisfy the project requirements.

In [5]:
df["Cancellation_Flag"] = np.where(df["Booking_Status"] == "Canceled", 1, 0)

df["Month"] = df["Date"].dt.month_name()
df["Month_Number"] = df["Date"].dt.month
df["Weekday"] = df["Date"].dt.day_name()

def rain_bin(rain):
    if rain == 0:
        return "No Rain"
    elif rain < 5:
        return "Light Rain"
    elif rain < 15:
        return "Moderate Rain"
    else:
        return "Heavy Rain"

df["Rain_Bin"] = df["Daily_Rainfall_mm"].apply(rain_bin)

df["Heat_Day_Type"] = np.where(
    df["Max_Temperature_F"] > 90,
    "Heat Wave Day",
    "Standard Day"
)

df.head()

,Booking_ID,Date,Service_Type,Daily_Rainfall_mm,Max_Temperature_F,Booking_Status,Cancellation_Probability,Cancellation_Flag,Month,Month_Number,Weekday,Rain_Bin,Heat_Day_Type
0,B00001,2024-01-01,Ceramic Coating,0.0,56.0,Completed,0.047,0,January,1,Monday,No Rain,Standard Day
1,B00002,2024-01-01,Ceramic Coating,0.0,56.0,Completed,0.030,0,January,1,Monday,No Rain,Standard Day
2,B00003,2024-01-01,Ceramic Coating,0.0,56.0,Completed,0.058,0,January,1,Monday,No Rain,Standard Day
3,B00004,2024-01-01,Interior Restoration,0.0,56.0,Completed,0.032,0,January,1,Monday,No Rain,Standard Day
4,B00005,2024-01-01,Interior Restoration,0.0,56.0,Completed,0.049,0,January,1,Monday,No Rain,Standard Day


## Create daily service-level cancellation rates

This gives Tableau a clean daily cancellation-rate field for scatter plots, box plots, and trend analysis.

In [6]:
daily_rates = (
    df.groupby(["Date", "Service_Type"], as_index=False)
    .agg(
        Daily_Bookings=("Booking_ID", "count"),
        Daily_Cancellations=("Cancellation_Flag", "sum"),
        Daily_Rainfall_mm=("Daily_Rainfall_mm", "mean"),
        Max_Temperature_F=("Max_Temperature_F", "mean")
    )
)

daily_rates["Daily_Cancellation_Rate"] = (
    daily_rates["Daily_Cancellations"] / daily_rates["Daily_Bookings"]
).round(3)

df = df.merge(
    daily_rates[[
        "Date",
        "Service_Type",
        "Daily_Bookings",
        "Daily_Cancellations",
        "Daily_Cancellation_Rate"
    ]],
    on=["Date", "Service_Type"],
    how="left"
)

df.head()

,Booking_ID,Date,Service_Type,Daily_Rainfall_mm,Max_Temperature_F,Booking_Status,Cancellation_Probability,Cancellation_Flag,Month,Month_Number,Weekday,Rain_Bin,Heat_Day_Type,Daily_Bookings,Daily_Cancellations,Daily_Cancellation_Rate
0,B00001,2024-01-01,Ceramic Coating,0.0,56.0,Completed,0.047,0,January,1,Monday,No Rain,Standard Day,3,0,0.0
1,B00002,2024-01-01,Ceramic Coating,0.0,56.0,Completed,0.030,0,January,1,Monday,No Rain,Standard Day,3,0,0.0
2,B00003,2024-01-01,Ceramic Coating,0.0,56.0,Completed,0.058,0,January,1,Monday,No Rain,Standard Day,3,0,0.0
3,B00004,2024-01-01,Interior Restoration,0.0,56.0,Completed,0.032,0,January,1,Monday,No Rain,Standard Day,7,0,0.0
4,B00005,2024-01-01,Interior Restoration,0.0,56.0,Completed,0.049,0,January,1,Monday,No Rain,Standard Day,7,0,0.0


## Check dataset size and cancellation distribution

In [7]:
print(df.shape)
print(df["Booking_Status"].value_counts())
print(df["Service_Type"].value_counts())

(3582, 16)
Booking_Status
Completed    3277
Canceled      305
Name: count, dtype: int64
Service_Type
Interior Restoration    1970
Ceramic Coating         1612
Name: count, dtype: int64


## Check cancellation behavior by service type

In [8]:
pd.crosstab(
    df["Service_Type"],
    df["Booking_Status"],
    normalize="index"
).round(3)

Booking_Status,Canceled,Completed
Service_Type,,
Ceramic Coating,0.104,0.896
Interior Restoration,0.070,0.930


## Check cancellation behavior by rain bin

In [9]:
pd.crosstab(
    df["Rain_Bin"],
    df["Booking_Status"],
    normalize="index"
).round(3)

Booking_Status,Canceled,Completed
Rain_Bin,,
Heavy Rain,0.224,0.776
Light Rain,0.089,0.911
Moderate Rain,0.139,0.861
No Rain,0.062,0.938


## Check cancellation behavior by heat day type

In [10]:
pd.crosstab(
    df["Heat_Day_Type"],
    df["Booking_Status"],
    normalize="index"
).round(3)

Booking_Status,Canceled,Completed
Heat_Day_Type,,
Heat Wave Day,0.150,0.850
Standard Day,0.076,0.924


## Preview the final synthetic operations dataset

In [11]:
df[[
    "Booking_ID",
    "Date",
    "Service_Type",
    "Daily_Rainfall_mm",
    "Max_Temperature_F",
    "Rain_Bin",
    "Heat_Day_Type",
    "Booking_Status",
    "Cancellation_Flag",
    "Daily_Bookings",
    "Daily_Cancellations",
    "Daily_Cancellation_Rate"
]].head(20)

,Booking_ID,Date,Service_Type,Daily_Rainfall_mm,Max_Temperature_F,Rain_Bin,Heat_Day_Type,Booking_Status,Cancellation_Flag,Daily_Bookings,Daily_Cancellations,Daily_Cancellation_Rate
0,B00001,2024-01-01,Ceramic Coating,0.0,56.0,No Rain,Standard Day,Completed,0,3,0,0.000
1,B00002,2024-01-01,Ceramic Coating,0.0,56.0,No Rain,Standard Day,Completed,0,3,0,0.000
2,B00003,2024-01-01,Ceramic Coating,0.0,56.0,No Rain,Standard Day,Completed,0,3,0,0.000
3,B00004,2024-01-01,Interior Restoration,0.0,56.0,No Rain,Standard Day,Completed,0,7,0,0.000
4,B00005,2024-01-01,Interior Restoration,0.0,56.0,No Rain,Standard Day,Completed,0,7,0,0.000
5,B00006,2024-01-01,Interior Restoration,0.0,56.0,No Rain,Standard Day,Completed,0,7,0,0.000
6,B00007,2024-01-01,Interior Restoration,0.0,56.0,No Rain,Standard Day,Completed,0,7,0,0.000
7,B00008,2024-01-01,Interior Restoration,0.0,56.0,No Rain,Standard Day,Completed,0,7,0,0.000
8,B00009,2024-01-01,Interior Restoration,0.0,56.0,No Rain,Standard Day,Completed,0,7,0,0.000
9,B00010,2024-01-01,Interior Restoration,0.0,56.0,No Rain,Standard Day,Completed,0,7,0,0.000


## Validate the relationship between rainfall and cancellations

Ceramic Coating should show stronger cancellation sensitivity as rainfall increases.

In [12]:
rain_summary = (
    df.groupby(["Service_Type", "Rain_Bin"], as_index=False)
    .agg(
        Total_Bookings=("Booking_ID", "count"),
        Total_Cancellations=("Cancellation_Flag", "sum"),
        Avg_Cancellation_Rate=("Cancellation_Flag", "mean"),
        Avg_Rainfall_mm=("Daily_Rainfall_mm", "mean")
    )
)

rain_summary["Avg_Cancellation_Rate"] = rain_summary["Avg_Cancellation_Rate"].round(3)
rain_summary.sort_values(["Service_Type", "Avg_Rainfall_mm"])

,Service_Type,Rain_Bin,Total_Bookings,Total_Cancellations,Avg_Cancellation_Rate,Avg_Rainfall_mm
3,Ceramic Coating,No Rain,1094,58,0.053,0.000000
1,Ceramic Coating,Light Rain,175,25,0.143,3.324571
2,Ceramic Coating,Moderate Rain,288,62,0.215,8.816319
0,Ceramic Coating,Heavy Rain,55,23,0.418,19.461818
7,Interior Restoration,No Rain,1346,94,0.070,0.000000
5,Interior Restoration,Light Rain,197,8,0.041,3.343147
6,Interior Restoration,Moderate Rain,330,24,0.073,8.625152
4,Interior Restoration,Heavy Rain,97,11,0.113,20.005155


## Validate the relationship between heat and cancellations

Interior Restoration should show stronger cancellation sensitivity on heat-wave days.

In [13]:
heat_summary = (
    df.groupby(["Service_Type", "Heat_Day_Type"], as_index=False)
    .agg(
        Total_Bookings=("Booking_ID", "count"),
        Total_Cancellations=("Cancellation_Flag", "sum"),
        Avg_Cancellation_Rate=("Cancellation_Flag", "mean"),
        Avg_Temperature_F=("Max_Temperature_F", "mean")
    )
)

heat_summary["Avg_Cancellation_Rate"] = heat_summary["Avg_Cancellation_Rate"].round(3)
heat_summary.sort_values(["Service_Type", "Heat_Day_Type"])

,Service_Type,Heat_Day_Type,Total_Bookings,Total_Cancellations,Avg_Cancellation_Rate,Avg_Temperature_F
0,Ceramic Coating,Heat Wave Day,200,19,0.095,97.374000
1,Ceramic Coating,Standard Day,1412,149,0.106,66.970609
2,Interior Restoration,Heat Wave Day,261,50,0.192,96.973946
3,Interior Restoration,Standard Day,1709,87,0.051,67.126448


## Create the final Tableau-ready dataset

We keep both row-level booking fields and daily service-level cancellation fields so Tableau can support scatter plots, box plots, filters, and dashboard summaries.

In [14]:
final_columns = [
    "Booking_ID",
    "Date",
    "Service_Type",
    "Daily_Rainfall_mm",
    "Max_Temperature_F",
    "Rain_Bin",
    "Heat_Day_Type",
    "Booking_Status",
    "Cancellation_Flag",
    "Cancellation_Probability",
    "Daily_Bookings",
    "Daily_Cancellations",
    "Daily_Cancellation_Rate",
    "Month",
    "Month_Number",
    "Weekday"
]

final_df = df[final_columns].copy()

print(final_df.shape)
final_df.head()

(3582, 16)


,Booking_ID,Date,Service_Type,Daily_Rainfall_mm,Max_Temperature_F,Rain_Bin,Heat_Day_Type,Booking_Status,Cancellation_Flag,Cancellation_Probability,Daily_Bookings,Daily_Cancellations,Daily_Cancellation_Rate,Month,Month_Number,Weekday
0,B00001,2024-01-01,Ceramic Coating,0.0,56.0,No Rain,Standard Day,Completed,0,0.047,3,0,0.0,January,1,Monday
1,B00002,2024-01-01,Ceramic Coating,0.0,56.0,No Rain,Standard Day,Completed,0,0.030,3,0,0.0,January,1,Monday
2,B00003,2024-01-01,Ceramic Coating,0.0,56.0,No Rain,Standard Day,Completed,0,0.058,3,0,0.0,January,1,Monday
3,B00004,2024-01-01,Interior Restoration,0.0,56.0,No Rain,Standard Day,Completed,0,0.032,7,0,0.0,January,1,Monday
4,B00005,2024-01-01,Interior Restoration,0.0,56.0,No Rain,Standard Day,Completed,0,0.049,7,0,0.0,January,1,Monday


## Save the Tableau-ready CSV file

In [15]:
final_df.to_csv("mobile_detailing_weather_cancellations_ready.csv", index=False)
print("Saved as mobile_detailing_weather_cancellations_ready.csv")

Saved as mobile_detailing_weather_cancellations_ready.csv


## Download the final CSV in Google Colab

In [16]:
from google.colab import files
files.download("mobile_detailing_weather_cancellations_ready.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>